# Ireland Data Centre Energy Analytics

**Historical demand, grid context, forecasting and 2035 scenario analysis**

This notebook investigates how data-centre growth has changed Irish electricity demand and what continued growth could mean for the electricity system through 2035.

### Research questions
1. How quickly has metered electricity consumption by Irish data centres grown since 2015?
2. How has the data-centre share of total metered electricity changed?
3. What does the recent Irish electricity supply mix look like, including wind, gas, solar and net imports?
4. Which simple trend model performs best on recent unseen quarterly data?
5. How does the resulting 2035 trend projection compare with EirGrid planning scenarios?
6. What scale of additional electricity, renewable capacity and carbon exposure could the projection imply?

> **Important:** Forecasts and scenarios in this notebook are analytical projections, not official forecasts or predictions of future policy outcomes.

## Data sources and scope

Primary sources used in the project:

- **Central Statistics Office (CSO)** — *Data Centres Metered Electricity Consumption 2025*, table MEC02.  
  https://www.cso.ie/en/releasesandpublications/ep/p-dcmec/datacentresmeteredelectricityconsumption2025/
- **Central Statistics Office (CSO)** — *Electricity Supply by Source 2025*, tables ESSA02 and ESSA04.  
  https://www.cso.ie/en/releasesandpublications/ep/p-ess/electricitysupplybysource2025/
- **EirGrid** — *All-Island Resource Adequacy Assessment 2026–2035* and Ireland inputs/assumptions.  
  https://www.eirgrid.ie/airaa
- **Sustainable Energy Authority of Ireland (SEAI)** — electricity statistics, renewable capacity and conversion/emission factors.  
  https://www.seai.ie/data-and-insights/seai-statistics/

### Scope cautions
- CSO **metered consumption** and CSO **grid supply** are different statistical concepts. Grid supply includes generation, net imports and storage output and is not identical to end-user demand because of transmission/distribution losses and other accounting differences.
- EirGrid's **Data Centre and New Technology Load** category is broader than the CSO metered data-centre series and is expressed in **MVA**, not GWh. Comparisons are therefore made using **growth indexes**, not direct unit conversion.
- Scenario assumptions later in the notebook are explicitly labelled as illustrative sensitivities.

## 1. Environment and project paths

The setup below works with the existing Google Drive project in Colab and also supports a cloned repository when run locally. All raw files remain unchanged in `data/raw/`; generated datasets are written to `data/processed/`, and figures to `charts/`.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.holtwinters import Holt

warnings.filterwarnings("ignore", category=FutureWarning)

# Optional override for local/CI execution.
project_override = os.environ.get("IRELAND_DC_PROJECT_PATH")

if project_override:
    PROJECT_PATH = Path(project_override).expanduser().resolve()
else:
    in_colab = False
    try:
        from google.colab import drive
        in_colab = True
    except ImportError:
        pass

    if in_colab:
        drive.mount("/content/drive")
        drive_project = Path("/content/drive/MyDrive/Ireland Data Centre Energy Analytics")
        if drive_project.exists():
            PROJECT_PATH = drive_project
        else:
            raise FileNotFoundError(
                "Google Drive is mounted, but the project folder was not found. "
                "Set IRELAND_DC_PROJECT_PATH or place the project in MyDrive."
            )
    else:
        cwd = Path.cwd().resolve()
        if (cwd / "data" / "raw").exists():
            PROJECT_PATH = cwd
        elif (cwd.parent / "data" / "raw").exists():
            PROJECT_PATH = cwd.parent
        else:
            raise FileNotFoundError(
                "Could not locate data/raw. Run from the repository root/notebooks "
                "folder or set IRELAND_DC_PROJECT_PATH."
            )

RAW_DATA_PATH = PROJECT_PATH / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_PATH / "data" / "processed"
CHARTS_PATH = PROJECT_PATH / "charts"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
CHARTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Project path: {PROJECT_PATH}")

## 2. Load and validate the official datasets

In [ ]:
MEC02_FILE = RAW_DATA_PATH / "MEC02.20261111T151111.csv"
ESSA02_FILE = RAW_DATA_PATH / "ESSA02.20261111T151141.csv"
ESSA04_FILE = RAW_DATA_PATH / "ESSA04.20261111T151155.csv"

mec02 = pd.read_csv(MEC02_FILE, encoding="utf-8-sig")
essa02 = pd.read_csv(ESSA02_FILE, encoding="utf-8-sig")
essa04 = pd.read_csv(ESSA04_FILE, encoding="utf-8-sig")

print("MEC02:", mec02.shape)
print("ESSA02:", essa02.shape)
print("ESSA04:", essa04.shape)

In [ ]:
expected_mec02 = {
    "Statistic Label", "Quarter", "Electricity Consumption", "UNIT", "VALUE"
}
expected_essa02 = {"Statistic Label", "Month", "Supply", "UNIT", "VALUE"}
expected_essa04 = {"Statistic Label", "Year", "Generation", "UNIT", "VALUE"}

assert set(mec02.columns) == expected_mec02
assert set(essa02.columns) == expected_essa02
assert set(essa04.columns) == expected_essa04

assert mec02.isna().sum().sum() == 0
assert essa02.isna().sum().sum() == 0
assert essa04.isna().sum().sum() == 0

assert set(mec02["UNIT"]) == {"GWh"}
assert set(essa02["UNIT"]) == {"MWh"}
assert set(essa04["UNIT"]) == {"GWh"}

print("Schema, missing-value and unit checks passed.")
print("MEC02 period:", mec02["Quarter"].min(), "to", mec02["Quarter"].max())
print("ESSA02 months:", essa02["Month"].nunique())
print("ESSA04 years:", sorted(essa04["Year"].unique()))

## 3. Historical data-centre electricity demand, 2015–2025

MEC02 contains three quarterly series: total metered electricity consumption, data centres, and customers other than data centres. The raw long-format table is reshaped to one row per quarter and then aggregated to annual totals.

In [ ]:
quarterly = mec02.pivot(
    index="Quarter",
    columns="Electricity Consumption",
    values="VALUE"
).reset_index()
quarterly.columns.name = None

quarterly["Year"] = quarterly["Quarter"].str[:4].astype(int)
quarterly["Quarter_Number"] = quarterly["Quarter"].str[-1].astype(int)
quarterly["Data_Centre_Share_pct"] = (
    quarterly["Data centres"]
    / quarterly["All metered electricity consumption"]
    * 100
)
quarterly["QoQ_Growth_pct"] = quarterly["Data centres"].pct_change() * 100
quarterly["YoY_Quarterly_Growth_pct"] = quarterly["Data centres"].pct_change(4) * 100
quarterly["Rolling_4Q_GWh"] = quarterly["Data centres"].rolling(4).sum()

annual = (
    quarterly.groupby("Year", as_index=False)[
        [
            "All metered electricity consumption",
            "Data centres",
            "Customers other than data centres",
        ]
    ].sum()
)
annual["Data_Centre_Share_pct"] = (
    annual["Data centres"]
    / annual["All metered electricity consumption"]
    * 100
)
annual["YoY_Data_Centre_Growth_pct"] = annual["Data centres"].pct_change() * 100

display(annual)

In [ ]:
# Data-quality note: the quarterly MEC02 values sum to 7,662 GWh in 2025,
# while the CSO release headline reports 7,663 GWh. This 1 GWh difference
# is retained as a documented rounding/aggregation nuance rather than altered.
published_2025_gwh = 7663
dataset_2025_gwh = annual.loc[annual["Year"] == 2025, "Data centres"].iloc[0]

print(f"Quarterly-series 2025 total: {dataset_2025_gwh:,.0f} GWh")
print(f"Published CSO 2025 headline: {published_2025_gwh:,.0f} GWh")
print(f"Difference: {dataset_2025_gwh - published_2025_gwh:+,.0f} GWh")

In [ ]:
start = annual.iloc[0]
end = annual.iloc[-1]
years_between = int(end["Year"] - start["Year"])

cagr = ((end["Data centres"] / start["Data centres"]) ** (1 / years_between) - 1) * 100

dc_growth_gwh = end["Data centres"] - start["Data centres"]
total_growth_gwh = (
    end["All metered electricity consumption"]
    - start["All metered electricity consumption"]
)
dc_contribution_pct = dc_growth_gwh / total_growth_gwh * 100

headline_metrics = pd.DataFrame({
    "Metric": [
        "2015 data-centre consumption (GWh)",
        "2025 data-centre consumption (GWh)",
        "2015–2025 growth multiple",
        "2015–2025 CAGR (%)",
        "2015 data-centre share (%)",
        "2025 data-centre share (%)",
        "Share of total metered-demand growth attributable to data centres (%)",
    ],
    "Value": [
        start["Data centres"],
        end["Data centres"],
        end["Data centres"] / start["Data centres"],
        cagr,
        start["Data_Centre_Share_pct"],
        end["Data_Centre_Share_pct"],
        dc_contribution_pct,
    ],
})

display(headline_metrics.round(2))

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(annual["Year"], annual["Data centres"], marker="o", linewidth=2.5)
plt.title("Ireland Data Centre Electricity Consumption, 2015–2025", fontsize=16, weight="bold")
plt.xlabel("Year")
plt.ylabel("Electricity Consumption (GWh)")
plt.grid(alpha=0.25)
for x, y in zip(annual["Year"], annual["Data centres"]):
    plt.annotate(f"{y:,.0f}", (x, y), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "01_data_centre_consumption_2015_2025.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(annual["Year"], annual["Data_Centre_Share_pct"], marker="o", linewidth=2.5)
ax.set_title("Data Centres as a Share of Ireland's Metered Electricity", fontsize=16, weight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Share of Metered Electricity (%)")
ax.grid(alpha=0.25)
for x, y in zip(annual["Year"], annual["Data_Centre_Share_pct"]):
    ax.annotate(f"{y:.1f}%", (x, y), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(CHARTS_PATH / "02_data_centre_electricity_share.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(annual["Year"], annual["Data centres"], marker="o", label="Data Centres")
plt.plot(
    annual["Year"],
    annual["Customers other than data centres"],
    marker="o",
    label="Other Customers",
)
plt.title("Electricity Consumption: Data Centres vs Other Customers", fontsize=16, weight="bold")
plt.xlabel("Year")
plt.ylabel("Electricity Consumption (GWh)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "03_data_centres_vs_other_customers.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
heatmap_data = quarterly.pivot(index="Year", columns="Quarter_Number", values="Data centres")

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(heatmap_data.values, aspect="auto")
ax.set_title("Data Centre Electricity Consumption by Year and Quarter", fontsize=15, weight="bold")
ax.set_xlabel("Quarter")
ax.set_ylabel("Year")
ax.set_xticks(range(4), ["Q1", "Q2", "Q3", "Q4"])
ax.set_yticks(range(len(heatmap_data.index)), heatmap_data.index)

for i in range(len(heatmap_data.index)):
    for j in range(4):
        ax.text(j, i, f"{heatmap_data.iloc[i, j]:,.0f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax, label="GWh")
fig.tight_layout()
fig.savefig(CHARTS_PATH / "04_quarterly_consumption_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

### Historical interpretation

The data-centre series shows persistent structural growth rather than a simple seasonal pattern. Electricity consumption increased from 1,240 GWh in 2015 to 7,662 GWh when the quarterly 2025 observations are summed. The sector's share of metered electricity rose from roughly 5% to more than 23% over the same period. The growth rate moderated after the exceptional expansion around 2020–2022, but absolute consumption continued to reach new highs.

## 4. Irish electricity supply context

ESSA04 provides annual grid-supply data for 2023–2025. ESSA02 provides monthly 2025 supply by source. These data are used to understand the recent supply mix and seasonality, not to create a 2015–2025 grid history.

In [ ]:
grid_annual = essa04.pivot(index="Year", columns="Generation", values="VALUE").reset_index()
grid_annual.columns.name = None

grid_annual["Core_Renewables_GWh"] = (
    grid_annual["Wind"]
    + grid_annual["Solar"]
    + grid_annual["Renewable Hydro"]
    + grid_annual["Other Renewable"]
)
grid_annual["Wind_Share_Generation_pct"] = grid_annual["Wind"] / grid_annual["Total Generation"] * 100
grid_annual["Gas_Share_Generation_pct"] = grid_annual["Gas"] / grid_annual["Total Generation"] * 100
grid_annual["Net_Import_Share_Supply_pct"] = grid_annual["Net Imports"] / grid_annual["Total Electricity Supplied"] * 100

display(grid_annual.round(2))

In [ ]:
essa02 = essa02.copy()
essa02["Date"] = pd.to_datetime(essa02["Month"], format="%Y %B")

grid_monthly = essa02.pivot(index="Date", columns="Supply", values="VALUE").reset_index()
grid_monthly.columns.name = None

numeric_columns = grid_monthly.columns.drop("Date")
grid_monthly[numeric_columns] = grid_monthly[numeric_columns] / 1000  # MWh -> GWh

grid_monthly["Net Imports"] = grid_monthly["Imports"] + grid_monthly["Exports"]
grid_monthly["Storage Discharge"] = (
    grid_monthly["Pump Storage Discharging"] + grid_monthly["Battery Discharging"]
)
grid_monthly["Total Electricity Supplied"] = (
    grid_monthly["Total Generation"]
    + grid_monthly["Net Imports"]
    + grid_monthly["Storage Discharge"]
)

monthly_2025_total = grid_monthly["Total Electricity Supplied"].sum()
annual_2025_total = grid_annual.loc[
    grid_annual["Year"] == 2025, "Total Electricity Supplied"
].iloc[0]

print(f"Monthly-derived 2025 supply: {monthly_2025_total:,.3f} GWh")
print(f"Official annual 2025 supply: {annual_2025_total:,.3f} GWh")
print(f"Difference: {monthly_2025_total - annual_2025_total:+,.3f} GWh")

In [ ]:
grid_annual["Other_Generation"] = (
    grid_annual["Total Generation"]
    - grid_annual["Gas"]
    - grid_annual["Wind"]
    - grid_annual["Solar"]
)

supply_composition = grid_annual.set_index("Year")[[
    "Gas", "Wind", "Solar", "Other_Generation", "Net Imports", "Total Storage"
]]

ax = supply_composition.plot(kind="bar", stacked=True, figsize=(12, 7))
ax.set_title("Ireland Electricity Supply Composition, 2023–2025", fontsize=16, weight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Electricity Supplied (GWh)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Supply Source", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "05_ireland_electricity_supply_composition.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. 2025 cross-dataset analysis

The monthly grid data are aggregated to quarters and joined to the quarterly data-centre series. Because only four quarterly observations are available for the grid mix in 2025, this section is descriptive; it does **not** use four-point correlations to make causal claims.

In [ ]:
grid_monthly["Year"] = grid_monthly["Date"].dt.year
grid_monthly["Quarter_Number"] = grid_monthly["Date"].dt.quarter
grid_monthly["Quarter"] = (
    grid_monthly["Year"].astype(str) + "Q" + grid_monthly["Quarter_Number"].astype(str)
)

grid_quarterly = grid_monthly.groupby("Quarter", as_index=False)[[
    "Gas", "Wind", "Solar", "Renewable Hydro", "Imports", "Exports",
    "Net Imports", "Storage Discharge", "Total Generation", "Total Electricity Supplied"
]].sum()

dc_2025 = quarterly.loc[quarterly["Year"] == 2025, [
    "Quarter", "Data centres", "All metered electricity consumption", "Data_Centre_Share_pct"
]].copy()

combined_2025 = pd.merge(dc_2025, grid_quarterly, on="Quarter", how="inner", validate="one_to_one")
combined_2025["Wind_Share_Generation_pct"] = combined_2025["Wind"] / combined_2025["Total Generation"] * 100
combined_2025["Gas_Share_Generation_pct"] = combined_2025["Gas"] / combined_2025["Total Generation"] * 100
combined_2025["Net_Import_Share_Supply_pct"] = combined_2025["Net Imports"] / combined_2025["Total Electricity Supplied"] * 100

display(combined_2025.round(2))

In [ ]:
indexed = combined_2025[["Quarter", "Data centres", "Gas", "Wind", "Solar", "Net Imports"]].copy()
metrics = ["Data centres", "Gas", "Wind", "Solar", "Net Imports"]

for column in metrics:
    indexed[column] = indexed[column] / indexed[column].iloc[0] * 100

plt.figure(figsize=(12, 7))
for column in metrics:
    plt.plot(indexed["Quarter"], indexed[column], marker="o", linewidth=2, label=column)
plt.axhline(100, linewidth=1, linestyle="--")
plt.title("Quarterly Electricity Trends — Indexed to Q1 2025 = 100", fontsize=16, weight="bold")
plt.xlabel("Quarter")
plt.ylabel("Index (Q1 = 100)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "06_indexed_grid_trends_2025.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Forecasting data-centre electricity demand

### Validation design

The final notebook uses **rolling-origin one-step-ahead validation** over 2024 Q1–2025 Q4. For every test quarter, each model is refit using only information that would have been available before that quarter. This makes the seasonal-naive, linear, exponential and Holt models directly comparable and avoids leakage from later test observations.

Metrics:
- **MAE** — mean absolute error
- **RMSE** — root mean squared error
- **MAPE** — mean absolute percentage error
- **R²** — goodness-of-fit measure on the eight held-out observations

In [ ]:
forecast_df = quarterly[["Quarter", "Data centres"]].copy()
forecast_df = forecast_df.rename(columns={"Data centres": "Data_Centre_GWh"})
forecast_df["t"] = np.arange(len(forecast_df))

test_start = len(forecast_df) - 8
print("Validation period:", forecast_df.iloc[test_start]["Quarter"], "to", forecast_df.iloc[-1]["Quarter"])

In [ ]:
def evaluate_model(actual, predicted, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        "Model": model_name,
        "MAE_GWh": mean_absolute_error(actual, predicted),
        "RMSE_GWh": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE_pct": np.mean(np.abs((actual - predicted) / actual)) * 100,
        "R2": r2_score(actual, predicted),
    }

predictions = {
    "Seasonal Naive": [],
    "Linear Trend": [],
    "Exponential Trend": [],
    "Holt Damped Trend": [],
}
actual_values = []
validation_quarters = []

for idx in range(test_start, len(forecast_df)):
    history = forecast_df.iloc[:idx].copy()
    target = forecast_df.iloc[idx]

    validation_quarters.append(target["Quarter"])
    actual_values.append(target["Data_Centre_GWh"])

    # Same quarter in the previous year.
    predictions["Seasonal Naive"].append(forecast_df.iloc[idx - 4]["Data_Centre_GWh"])

    linear_model = LinearRegression().fit(history[["t"]], history["Data_Centre_GWh"])
    predictions["Linear Trend"].append(
        linear_model.predict(pd.DataFrame({"t": [target["t"]]}))[0]
    )

    exponential_model = LinearRegression().fit(
        history[["t"]], np.log(history["Data_Centre_GWh"])
    )
    predictions["Exponential Trend"].append(
        np.exp(exponential_model.predict(pd.DataFrame({"t": [target["t"]]}))[0])
    )

    holt_fit = Holt(
        history["Data_Centre_GWh"],
        damped_trend=True,
        initialization_method="estimated",
    ).fit(optimized=True)
    predictions["Holt Damped Trend"].append(float(holt_fit.forecast(1).iloc[0]))

model_results = pd.DataFrame([
    evaluate_model(actual_values, values, name)
    for name, values in predictions.items()
]).sort_values("RMSE_GWh")

display(model_results.round(3))

In [ ]:
validation_plot = pd.DataFrame({"Quarter": validation_quarters, "Actual": actual_values})
for model_name, values in predictions.items():
    validation_plot[model_name] = values

plt.figure(figsize=(13, 7))
plt.plot(validation_plot["Quarter"], validation_plot["Actual"], marker="o", linewidth=3, label="Actual")
for model_name in predictions:
    plt.plot(validation_plot["Quarter"], validation_plot[model_name], marker="o", linestyle="--", label=model_name)
plt.title("Rolling-Origin Forecast Model Validation, 2024–2025", fontsize=16, weight="bold")
plt.xlabel("Quarter")
plt.ylabel("Electricity Consumption (GWh)")
plt.legend()
plt.grid(alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "07_forecast_model_validation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
best_model_name = model_results.iloc[0]["Model"]
assert best_model_name == "Holt Damped Trend", "Re-check model selection if the dataset changes."

final_fit = Holt(
    forecast_df["Data_Centre_GWh"],
    damped_trend=True,
    initialization_method="estimated",
).fit(optimized=True)

future_values = final_fit.forecast(40)
future_quarters = [f"{year}Q{quarter}" for year in range(2026, 2036) for quarter in range(1, 5)]

future_forecast = pd.DataFrame({
    "Quarter": future_quarters,
    "Forecast_Data_Centre_GWh": future_values.values,
})
future_forecast["Year"] = future_forecast["Quarter"].str[:4].astype(int)

annual_forecast = future_forecast.groupby("Year", as_index=False)["Forecast_Data_Centre_GWh"].sum()
annual_forecast["Forecast_YoY_Growth_pct"] = annual_forecast["Forecast_Data_Centre_GWh"].pct_change() * 100

display(annual_forecast.round(2))

In [ ]:
plt.figure(figsize=(15, 7))
plt.plot(forecast_df["Quarter"], forecast_df["Data_Centre_GWh"], linewidth=2.5, label="Historical")
plt.plot(future_forecast["Quarter"], future_forecast["Forecast_Data_Centre_GWh"], linewidth=2.5, linestyle="--", label="Holt trend projection")
plt.axvline(x="2025Q4", linestyle="--", alpha=0.6)
plt.title("Ireland Data Centre Electricity Consumption — Historical and Trend Projection", fontsize=16, weight="bold")
plt.xlabel("Quarter")
plt.ylabel("Electricity Consumption (GWh)")
all_quarters = forecast_df["Quarter"].tolist() + future_forecast["Quarter"].tolist()
plt.xticks(range(0, len(all_quarters), 4), all_quarters[::4], rotation=45)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "08_data_centre_forecast_2035.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Benchmark against EirGrid planning scenarios

EirGrid's AIRAA 2026–2035 models **Data Centre and New Technology Load**, including large dedicated high-voltage connections. Its 2024 baseline is 959 MVA and its 2035 scenarios are 1,402 MVA (Low), 1,870 MVA (Median) and 2,183 MVA (High).

Because EirGrid uses **MVA** and a broader customer category while this notebook forecasts CSO metered electricity consumption in **GWh**, the comparison below normalises each series to **2024 = 100**. It is a growth benchmark, not a direct equivalence of demand units.

In [ ]:
eirgrid_2024_mva = 959

eirgrid_scenarios = pd.DataFrame({
    "Scenario": ["Low", "Median", "High"],
    "2035_Demand_MVA": [1402, 1870, 2183],
})
eirgrid_scenarios["2024_Demand_MVA"] = eirgrid_2024_mva
eirgrid_scenarios["Growth_2024_2035_pct"] = (
    eirgrid_scenarios["2035_Demand_MVA"] / eirgrid_scenarios["2024_Demand_MVA"] - 1
) * 100
eirgrid_scenarios["Index_2024_100"] = (
    eirgrid_scenarios["2035_Demand_MVA"] / eirgrid_scenarios["2024_Demand_MVA"] * 100
)

actual_2024_gwh = annual.loc[annual["Year"] == 2024, "Data centres"].iloc[0]
forecast_2035_gwh = annual_forecast.loc[
    annual_forecast["Year"] == 2035, "Forecast_Data_Centre_GWh"
].iloc[0]

our_growth_pct = (forecast_2035_gwh / actual_2024_gwh - 1) * 100
our_index = forecast_2035_gwh / actual_2024_gwh * 100

comparison = pd.concat([
    eirgrid_scenarios[["Scenario", "Growth_2024_2035_pct", "Index_2024_100"]].iloc[:2],
    pd.DataFrame({
        "Scenario": ["Holt Trend Projection"],
        "Growth_2024_2035_pct": [our_growth_pct],
        "Index_2024_100": [our_index],
    }),
    eirgrid_scenarios[["Scenario", "Growth_2024_2035_pct", "Index_2024_100"]].iloc[2:],
], ignore_index=True)

display(comparison.round(2))

In [ ]:
plt.figure(figsize=(11, 6))
bars = plt.bar(comparison["Scenario"], comparison["Index_2024_100"])
plt.axhline(100, linestyle="--", linewidth=1)
plt.title("2035 Demand Growth Benchmark — 2024 = 100", fontsize=16, weight="bold")
plt.ylabel("Index (2024 = 100)")
plt.xlabel("Scenario")
for bar, value in zip(bars, comparison["Index_2024_100"]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 3, f"{value:.0f}", ha="center", fontsize=10)
plt.grid(axis="y", alpha=0.25)
plt.xticks(rotation=8)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "09_forecast_vs_eirgrid_scenarios.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. 2035 electricity-system scenario analysis

The forecast is converted into an incremental energy requirement relative to 2025. Three illustrative supply mixes then stress-test how that additional electricity could be distributed between renewables, domestic flexible supply and net imports.

These supply mixes are **not** forecasts or policy recommendations; they are sensitivity scenarios used to make the scale of the additional energy requirement easier to interpret.

In [ ]:
baseline_2025_gwh = annual.loc[annual["Year"] == 2025, "Data centres"].iloc[0]
forecast_2030_gwh = annual_forecast.loc[
    annual_forecast["Year"] == 2030, "Forecast_Data_Centre_GWh"
].iloc[0]
additional_2035_gwh = forecast_2035_gwh - baseline_2025_gwh

scenario_summary = pd.DataFrame({
    "Year": [2025, 2030, 2035],
    "Data_Centre_GWh": [baseline_2025_gwh, forecast_2030_gwh, forecast_2035_gwh],
})
scenario_summary["Additional_vs_2025_GWh"] = scenario_summary["Data_Centre_GWh"] - baseline_2025_gwh
scenario_summary["Average_Load_MW"] = scenario_summary["Data_Centre_GWh"] / 8.76
scenario_summary["Additional_Average_Load_MW"] = scenario_summary["Additional_vs_2025_GWh"] / 8.76

display(scenario_summary.round(1))

In [ ]:
system_context = pd.DataFrame({
    "Reference": [
        "2025 Total Grid Supply",
        "2025 Wind Generation",
        "2025 Gas Generation",
        "2025 Net Imports",
    ],
    "GWh": [35884, 11782, 13918, 6130],
})
system_context["2035_Increment_as_pct"] = additional_2035_gwh / system_context["GWh"] * 100

display(system_context.round(1))

In [ ]:
supply_scenarios = pd.DataFrame({
    "Scenario": ["Renewables-led", "Balanced", "Import-heavy"],
    "Renewables_pct": [70, 50, 35],
    "Domestic_Flexible_pct": [20, 30, 25],
    "Net_Imports_pct": [10, 20, 40],
})
supply_scenarios["Total_pct"] = supply_scenarios[[
    "Renewables_pct", "Domestic_Flexible_pct", "Net_Imports_pct"
]].sum(axis=1)
assert (supply_scenarios["Total_pct"] == 100).all()

scenario_2035 = supply_scenarios.copy()
scenario_2035["Additional_Demand_GWh"] = additional_2035_gwh
for source, pct_col in [
    ("Renewables_GWh", "Renewables_pct"),
    ("Domestic_Flexible_GWh", "Domestic_Flexible_pct"),
    ("Net_Imports_GWh", "Net_Imports_pct"),
]:
    scenario_2035[source] = scenario_2035["Additional_Demand_GWh"] * scenario_2035[pct_col] / 100

display(scenario_2035.round(1))

In [ ]:
plot_data = scenario_2035.set_index("Scenario")[[
    "Renewables_GWh", "Domestic_Flexible_GWh", "Net_Imports_GWh"
]]
ax = plot_data.plot(kind="bar", stacked=True, figsize=(12, 7))
ax.set_title("Illustrative Supply Scenarios for Additional Data Centre Demand — 2035", fontsize=15, weight="bold")
ax.set_xlabel("Scenario")
ax.set_ylabel("Additional Electricity Required (GWh)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Supply Source", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "10_2035_supply_scenarios.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Illustrative renewable-capacity requirement

SEAI reports a 2024 Irish wind capacity factor of approximately 29.2%. An indicative solar capacity factor is derived here from SEAI's reported end-2023/end-2024 installed solar capacity and 2024 solar generation. Because capacity was added throughout the year, the solar result is an approximation rather than an official SEAI capacity-factor statistic.

The calculation answers a scale question: if 70% of the incremental 2035 demand were supplied by renewables, and that renewable energy were split 70% wind / 30% solar, what additional installed capacity would be implied under these performance assumptions?

In [ ]:
WIND_CAPACITY_FACTOR = 0.292
solar_capacity_2023_mw = 783.79
solar_capacity_2024_mw = 1323.40
solar_generation_2024_gwh = 1090

average_solar_capacity_2024_mw = (solar_capacity_2023_mw + solar_capacity_2024_mw) / 2
SOLAR_CAPACITY_FACTOR = solar_generation_2024_gwh / (average_solar_capacity_2024_mw * 8.76)

def required_capacity_mw(annual_energy_gwh, capacity_factor):
    return annual_energy_gwh / (8.76 * capacity_factor)

renewable_energy_2035 = scenario_2035.loc[
    scenario_2035["Scenario"] == "Renewables-led", "Renewables_GWh"
].iloc[0]

wind_energy_gwh = renewable_energy_2035 * 0.70
solar_energy_gwh = renewable_energy_2035 * 0.30

renewable_capacity_mix = pd.DataFrame({
    "Technology": ["Wind", "Solar"],
    "Energy_GWh": [wind_energy_gwh, solar_energy_gwh],
    "Capacity_Factor": [WIND_CAPACITY_FACTOR, SOLAR_CAPACITY_FACTOR],
    "Required_Capacity_MW": [
        required_capacity_mw(wind_energy_gwh, WIND_CAPACITY_FACTOR),
        required_capacity_mw(solar_energy_gwh, SOLAR_CAPACITY_FACTOR),
    ],
})

display(renewable_capacity_mix.round(2))

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(renewable_capacity_mix["Technology"], renewable_capacity_mix["Required_Capacity_MW"])
plt.title("Illustrative Additional Renewable Capacity for 2035 Data Centre Demand", fontsize=15, weight="bold")
plt.ylabel("Additional Installed Capacity (MW)")
plt.xlabel("Technology")
for bar, value in zip(bars, renewable_capacity_mix["Required_Capacity_MW"]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 25, f"{value:,.0f} MW", ha="center")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "11_additional_renewable_capacity_2035.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Carbon-intensity sensitivity

SEAI's provisional 2025 electricity-consumption factor is **197.8 gCO₂/kWh** for emissions arising within Ireland. The 100 and 50 gCO₂/kWh cases below are analytical sensitivities, not official 2035 forecasts.

SEAI notes that the consumption factor excludes emissions arising outside Ireland from imported electricity generation. Therefore these results should be interpreted as **Irish grid-emissions sensitivities**, not full lifecycle carbon footprints.

In [ ]:
carbon_scenarios = pd.DataFrame({
    "Scenario": ["2025 Grid Intensity", "Moderate Decarbonisation", "Deep Decarbonisation"],
    "Carbon_Intensity_gCO2_kWh": [197.8, 100.0, 50.0],
})

# 1 GWh × 1 gCO2/kWh = 1 tonne CO2.
carbon_scenarios["Incremental_2035_Emissions_tCO2"] = (
    additional_2035_gwh * carbon_scenarios["Carbon_Intensity_gCO2_kWh"]
)
carbon_scenarios["Incremental_2035_Emissions_MtCO2"] = (
    carbon_scenarios["Incremental_2035_Emissions_tCO2"] / 1_000_000
)
carbon_scenarios["Total_2035_Emissions_MtCO2"] = (
    forecast_2035_gwh * carbon_scenarios["Carbon_Intensity_gCO2_kWh"] / 1_000_000
)

display(carbon_scenarios.round(3))

In [ ]:
plt.figure(figsize=(11, 6))
bars = plt.bar(carbon_scenarios["Scenario"], carbon_scenarios["Incremental_2035_Emissions_MtCO2"])
plt.title("Carbon Sensitivity of Additional Data Centre Demand in 2035", fontsize=15, weight="bold")
plt.ylabel("Annual Emissions (MtCO₂)")
plt.xlabel("Grid Carbon-Intensity Scenario")
plt.xticks(rotation=10)
for bar, value in zip(bars, carbon_scenarios["Incremental_2035_Emissions_MtCO2"]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.03, f"{value:.2f} Mt", ha="center")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(CHARTS_PATH / "12_2035_carbon_intensity_scenarios.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Save reproducible processed outputs

In [ ]:
quarterly.to_csv(PROCESSED_DATA_PATH / "data_centre_electricity_quarterly.csv", index=False)
annual.to_csv(PROCESSED_DATA_PATH / "data_centre_electricity_annual.csv", index=False)
grid_annual.to_csv(PROCESSED_DATA_PATH / "electricity_supply_annual_2023_2025.csv", index=False)
grid_monthly.to_csv(PROCESSED_DATA_PATH / "electricity_supply_monthly_2025.csv", index=False)
combined_2025.to_csv(PROCESSED_DATA_PATH / "data_centres_grid_quarterly_2025.csv", index=False)
model_results.to_csv(PROCESSED_DATA_PATH / "forecast_model_evaluation.csv", index=False)
future_forecast.to_csv(PROCESSED_DATA_PATH / "data_centre_quarterly_forecast_2026_2035.csv", index=False)
annual_forecast.to_csv(PROCESSED_DATA_PATH / "data_centre_annual_forecast_2026_2035.csv", index=False)
comparison.to_csv(PROCESSED_DATA_PATH / "eirgrid_growth_benchmark_2035.csv", index=False)
scenario_summary.to_csv(PROCESSED_DATA_PATH / "data_centre_demand_scenario_summary.csv", index=False)
scenario_2035.to_csv(PROCESSED_DATA_PATH / "2035_electricity_supply_scenarios.csv", index=False)
renewable_capacity_mix.to_csv(PROCESSED_DATA_PATH / "2035_renewable_capacity_requirements.csv", index=False)
carbon_scenarios.to_csv(PROCESSED_DATA_PATH / "2035_data_centre_carbon_scenarios.csv", index=False)

print("Processed outputs saved to:", PROCESSED_DATA_PATH)
print("Charts saved to:", CHARTS_PATH)

## 12. Key findings

The notebook supports the following conclusions from the analysed datasets and modelling:

1. **Data-centre electricity demand has become a major component of Irish metered consumption.** The quarterly CSO series grows from 1,240 GWh in 2015 to 7,662 GWh in 2025, while its metered-electricity share rises from about 5% to more than 23%.
2. **Data centres explain most of the net growth in metered consumption over 2015–2025.** Their increase accounts for roughly three quarters of the total increase in the MEC02 series.
3. **Recent electricity-supply growth has relied increasingly on imports.** CSO ESSA04 shows total grid supply rising from 2023 to 2025 while domestic generation remains comparatively flat and net imports increase.
4. **The damped Holt trend is the strongest of the tested simple models on rolling-origin 2024–2025 validation.** This supports its use as a transparent statistical trend projection, while not eliminating long-horizon model risk.
5. **The 2035 trend projection lies between EirGrid's Median and High growth scenarios when both are indexed to 2024.** This is a growth comparison only; the underlying datasets and units differ.
6. **The incremental energy requirement is material.** The projection implies roughly 6.6 TWh/year more data-centre consumption in 2035 than in 2025, or around 0.76 GW of additional average continuous load.
7. **Decarbonisation materially changes the carbon exposure of that growth.** The same incremental demand has very different emissions implications under 197.8, 100 and 50 gCO₂/kWh grid-intensity sensitivities.

## 13. Limitations

- The data-centre series is an aggregate national metered-consumption series; it does not provide site-level drivers, efficiency, workload or PUE information.
- ESSA04 only covers 2023–2025 and ESSA02 only covers 2025 in the files used here, limiting historical grid-mix inference.
- Grid supply and consumer metered demand are not identical measures.
- The four-quarter 2025 cross-dataset section is descriptive and is not sufficient for causal inference.
- Long-horizon forecasts are sensitive to structural changes in planning, grid connections, technology, regulation and customer behaviour.
- EirGrid's MVA scenarios cover a broader high-voltage data-centre/new-technology category than the CSO GWh consumption series.
- Renewable-capacity requirements depend strongly on capacity-factor assumptions and the illustrative wind/solar split.
- Carbon sensitivities use an Irish electricity-consumption factor that excludes emissions from electricity generated abroad and imported into Ireland.

These limitations are part of the analysis rather than hidden assumptions: the purpose is to quantify plausible scale while clearly separating observed data, statistical projection and illustrative scenario modelling.